# JEPA Notebook Demo: V-JEPA Future Selection, Real-World Video, and Grounded Agent Commentary

This notebook keeps the existing Moving MNIST control benchmark intact, then extends the repo into a more realistic real-video future-selection demo. The system still uses frozen V-JEPA embeddings plus the engineered compatibility scorer as the primary decision mechanism, but now it can cache a lightweight real-world dataset subset, score candidate futures, generate grounded commentary, and run a small live agent loop one example at a time.

## 1. Environment and Runtime Checks

These cells attach the repository inside the Colab runtime, install dependencies conservatively, and confirm the GPU/runtime state before any model work begins.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DEFAULT_REPO_URL = 'https://github.com/Praveen-Rangavajhula/jepa-representation-learning.git'
REPO_ROOT_HINT = os.environ.get('JEPA_REPO_ROOT', '').strip()
REPO_URL = os.environ.get('JEPA_REPO_URL', DEFAULT_REPO_URL).strip()
REPO_BRANCH = os.environ.get('JEPA_REPO_BRANCH', '').strip()
AUTO_UPDATE_REPO = os.environ.get('JEPA_AUTO_UPDATE_REPO', '1').strip().lower() in {'1', 'true', 'yes'}
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

def looks_like_repo_root(path: Path) -> bool:
    return (path / 'src' / 'jepa').exists() and (path / 'notebooks').exists()

def discover_existing_repo() -> Path | None:
    candidates = []
    if REPO_ROOT_HINT:
        candidates.append(Path(REPO_ROOT_HINT).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    for root in (Path('/content'), Path('/content/drive/MyDrive')):
        if not root.exists():
            continue
        candidates.append(root)
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    candidates.append(child)
        except Exception:
            pass

    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except Exception:
            continue
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if looks_like_repo_root(candidate):
            return candidate
    return None

def update_repo_if_requested(repo_root: Path) -> Path:
    if not AUTO_UPDATE_REPO:
        return repo_root.resolve()
    if not str(repo_root).startswith('/content'):
        return repo_root.resolve()
    if not (repo_root / '.git').exists():
        return repo_root.resolve()

    if REPO_BRANCH:
        subprocess.check_call(['git', '-C', str(repo_root), 'checkout', REPO_BRANCH])
    print(f'Updating existing cloned repo at {repo_root}')
    subprocess.check_call(['git', '-C', str(repo_root), 'pull', '--ff-only'])
    return repo_root.resolve()

def clone_repo_if_needed() -> Path | None:
    if not REPO_URL:
        return None
    repo_name = Path(REPO_URL.rstrip('/')).stem
    if repo_name.endswith('.git'):
        repo_name = repo_name[:-4]
    target_root = Path('/content') / repo_name

    if looks_like_repo_root(target_root):
        print(f'Reusing existing cloned repo: {target_root}')
        return update_repo_if_requested(target_root)

    command = ['git', 'clone']
    if REPO_BRANCH:
        command.extend(['--branch', REPO_BRANCH])
    command.extend([REPO_URL, str(target_root)])
    print('Cloning repo:', ' '.join(command))
    subprocess.check_call(command)
    return target_root.resolve()

REPO_ROOT = discover_existing_repo()
if REPO_ROOT is not None:
    REPO_ROOT = update_repo_if_requested(REPO_ROOT)
if REPO_ROOT is None:
    REPO_ROOT = clone_repo_if_needed()

if REPO_ROOT is None:
    raise FileNotFoundError(
        'Could not find the repository in the Colab runtime. Set JEPA_REPO_ROOT or let the notebook clone it.'
    )

SRC_ROOT = REPO_ROOT / 'src'
assert (SRC_ROOT / 'jepa').exists(), f'Expected package directory at {SRC_ROOT / "jepa"}'

print(f'Repository root: {REPO_ROOT}')
print(f'Source root: {SRC_ROOT}')
print(f'Auto-update existing clone: {AUTO_UPDATE_REPO}')
%cd $REPO_ROOT
print(f'Python executable: {sys.executable}')
print(f'Working directory: {Path.cwd()}')

In [ ]:
import importlib.util
import subprocess
import sys

requirements_file = REPO_ROOT / 'requirements-colab.txt'
core_specs = {
    'torch': 'torch>=2.6,<2.8',
    'torchvision': 'torchvision>=0.21,<0.23',
}
USE_TORCH_HUB_FALLBACK = os.environ.get('JEPA_USE_TORCH_HUB_FALLBACK', '0').strip().lower() in {'1', 'true', 'yes'}
missing_core = [name for name in core_specs if importlib.util.find_spec(name) is None]

if missing_core:
    print('Installing missing core packages:', missing_core)
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', *[core_specs[name] for name in missing_core]],
        cwd=str(REPO_ROOT),
    )
else:
    print('Core packages already available: torch, torchvision')

print(f'Installing notebook dependencies from {requirements_file.name} (safe to rerun).')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)], cwd=str(REPO_ROOT))

if USE_TORCH_HUB_FALLBACK:
    print('Installing fallback torch.hub extras: timm, einops')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'timm', 'einops'], cwd=str(REPO_ROOT))

print('If this cell upgraded already-imported packages, restart the runtime once before continuing.')

In [ ]:
from importlib import metadata as importlib_metadata
import json
import torch

HF_CACHE_DIR = Path(os.environ.get('HF_HOME', '/content/.cache/huggingface'))
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

versions = {}
for name in ['torch', 'torchvision', 'transformers', 'accelerate', 'huggingface_hub', 'safetensors']:
    try:
        versions[name] = importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        versions[name] = None

print('Runtime versions:')
print(json.dumps(versions, indent=2))
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'Current device: {torch.cuda.get_device_name(0)}')
print(f'Hugging Face cache dir: {HF_CACHE_DIR}')
print(f'Fallback torch.hub enabled: {USE_TORCH_HUB_FALLBACK}')

## 2. Moving MNIST Control Benchmark

Moving MNIST remains the cheap, controlled benchmark. These cells still build 16-frame sequences, split them into observed prefixes and candidate futures, and save a few visual artifacts so we can compare the real-video path against a fully known synthetic setup.

In [ ]:
import importlib
import sys

if str(SRC_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT.resolve()))
importlib.invalidate_caches()

from jepa.commentary import DeterministicCommentaryGenerator, LLMReadyCommentaryBuilder
from jepa.data import (
    MovingMNISTConfig,
    RealVideoDataError,
    RealVideoDependencyError,
    RealVideoSubsetConfig,
    SomethingSomethingV2SubsetAdapter,
    available_real_video_templates,
    create_moving_mnist_dataloaders,
    save_sample_visualizations,
    summarize_batch,
)
from jepa.llm import build_default_commentary_service, generate_commentary
from jepa.agents import run_future_selection_pipeline
from jepa.models import VJEPA2Adapter, VJEPA2AdapterConfig, make_generic_vjepa_config, make_ssv2_vjepa_config
from jepa.scoring import VJEPAFutureScorer
from jepa.tasks import (
    FutureSelectionConfig,
    FutureSelectionDataset,
    save_future_selection_examples,
    summarize_future_selection_example,
)
from jepa.tools import (
    build_baseline_comparison_table,
    build_candidate_score_table,
    build_future_selection_context,
    build_multi_evaluator_comparison_table,
    build_ranking_summary,
    run_future_selection_benchmark,
    save_future_selection_benchmark_artifacts,
)

moving_config = MovingMNISTConfig(
    sequence_length=16,
    image_size=64,
    num_digits=2,
    velocity_range=(1.5, 3.5),
    train_size=512,
    test_size=128,
    mnist_root=str(REPO_ROOT / 'data'),
    seed=42,
)

task_config = FutureSelectionConfig(
    sequence_length=16,
    observed_length=8,
    future_length=8,
    num_candidates=4,
    seed=7,
)

dataset_artifacts = create_moving_mnist_dataloaders(
    moving_config,
    batch_size=8,
    num_workers=0,
    pin_memory=False,
)
train_dataset = dataset_artifacts['train_dataset']
test_dataset = dataset_artifacts['test_dataset']
train_loader = dataset_artifacts['train_loader']

train_task_dataset = FutureSelectionDataset(train_dataset, config=task_config)
test_task_dataset = FutureSelectionDataset(test_dataset, config=task_config)
example = train_task_dataset[0]
moving_batch = next(iter(train_loader))

real_video_config = RealVideoSubsetConfig(
    cache_root=os.environ.get('JEPA_REAL_VIDEO_CACHE_ROOT', 'data/real_video_cache'),
    source_window_length=int(os.environ.get('JEPA_REAL_VIDEO_SOURCE_WINDOW_LENGTH', '32')),
    image_size=int(os.environ.get('JEPA_REAL_VIDEO_IMAGE_SIZE', '256')),
    train_examples_per_template=int(os.environ.get('JEPA_REAL_VIDEO_TRAIN_PER_TEMPLATE', '24')),
    eval_examples_per_template=int(os.environ.get('JEPA_REAL_VIDEO_EVAL_PER_TEMPLATE', '16')),
    confident_eval_examples_per_template=int(os.environ.get('JEPA_REAL_VIDEO_CONFIDENT_PER_TEMPLATE', '40')),
    max_source_scan=int(os.environ.get('JEPA_REAL_VIDEO_MAX_SCAN', '20000')),
    local_manifest_path=os.environ.get('JEPA_REAL_VIDEO_MANIFEST') or None,
    streaming=os.environ.get('JEPA_REAL_VIDEO_STREAMING', '0').strip().lower() in {'1', 'true', 'yes'},
)

moving_results_dir = REPO_ROOT / 'results' / 'moving_mnist'
task_results_dir = REPO_ROOT / 'results' / 'task_examples'
vjepa_results_dir = REPO_ROOT / 'results' / 'vjepa_eval'
real_video_results_dir = REPO_ROOT / 'results' / 'real_video_eval'
reranker_results_dir = REPO_ROOT / 'results' / 'real_video_reranker_eval'
agent_live_results_dir = REPO_ROOT / 'results' / 'agent_live'
artifact_export_dir = REPO_ROOT / 'results' / 'artifact_exports'
report_dir = REPO_ROOT / 'report'
for path in (
    moving_results_dir,
    task_results_dir,
    vjepa_results_dir,
    real_video_results_dir,
    reranker_results_dir,
    agent_live_results_dir,
    artifact_export_dir,
    report_dir,
):
    path.mkdir(parents=True, exist_ok=True)

moving_paths = save_sample_visualizations(train_dataset, moving_results_dir, sample_indices=(0, 1), max_frames=8)
task_paths = save_future_selection_examples([train_task_dataset[i] for i in range(2)], task_results_dir, max_examples=2)

commentary_generator = DeterministicCommentaryGenerator()
llm_ready_builder = LLMReadyCommentaryBuilder()
commentary_service = build_default_commentary_service()

print('Moving MNIST batch summary:')
print(summarize_batch(moving_batch))
print('\nTask example summary:')
print(summarize_future_selection_example(example))
print('\nConfigured real-video templates:')
for template in available_real_video_templates(real_video_config):
    print(f'- {template}')
print('\nSaved data artifacts:')
for path in [*moving_paths, *task_paths]:
    print(f'- {path}')


## 3. V-JEPA 2 Loading and Preprocessing

These cells load the official V-JEPA checkpoint, inspect how Moving MNIST clips are converted into the model's expected RGB / 64-frame format, and confirm that embeddings can be extracted before any ranking or commentary is attempted.

In [ ]:
import json

generic_model_id = os.environ.get('JEPA_VJEPA_MODEL_ID')
adapter_config = make_generic_vjepa_config(
    model_id=generic_model_id or None,
    backend='huggingface',
    fallback_backend='torch_hub',
    fallback_model_name='vjepa2_vit_large',
    cache_dir=str(HF_CACHE_DIR),
)
adapter = VJEPA2Adapter(adapter_config)
vjepa_scorer = VJEPAFutureScorer(adapter=adapter)
print(f'V-JEPA scorer default variant: {vjepa_scorer.scoring_variant}')
print(f'Masked scorer signature: {vjepa_scorer.masked_runtime_signature}')
load_error = None
runtime_info = None

try:
    runtime_info = adapter.load()
    print(json.dumps(runtime_info, indent=2))
except Exception as exc:
    load_error = exc
    print(f'V-JEPA load failed: {exc}')
    if not USE_TORCH_HUB_FALLBACK:
        print('If needed, set JEPA_USE_TORCH_HUB_FALLBACK=1 and rerun the install cell for the official torch.hub fallback path.')

In [ ]:
import json
import torch

if load_error is not None:
    raise RuntimeError(f'Cannot continue preprocessing sanity checks because V-JEPA failed to load: {load_error}')

true_candidate = example.candidates[example.correct_index]
combined_true_clip = torch.cat([example.observed, true_candidate], dim=0)
observed_prep_summary = adapter.summarize_preprocessing(example.observed)
combined_prep_summary = adapter.summarize_preprocessing(combined_true_clip)

print('Observed clip preprocessing summary:')
print(json.dumps(observed_prep_summary, indent=2))
print('\nCombined observed+future clip preprocessing summary:')
print(json.dumps(combined_prep_summary, indent=2))

In [ ]:
import torch

if load_error is not None:
    raise RuntimeError(f'Cannot continue embedding sanity checks because V-JEPA failed to load: {load_error}')

observed_embedding = adapter.encode_video(example.observed)
true_future_embedding = adapter.encode_video(example.candidates[example.correct_index])
combined_embedding = adapter.encode_video(torch.cat([example.observed, example.candidates[example.correct_index]], dim=0))

print('Embedding sanity checks:')
print(f'- observed embedding shape: {tuple(observed_embedding.shape)} | norm={float(torch.linalg.norm(observed_embedding).item()):.4f}')
print(f'- true future embedding shape: {tuple(true_future_embedding.shape)} | norm={float(torch.linalg.norm(true_future_embedding).item()):.4f}')
print(f'- combined clip embedding shape: {tuple(combined_embedding.shape)} | norm={float(torch.linalg.norm(combined_embedding).item()):.4f}')

## 4. Moving MNIST Control Example

The engineered V-JEPA scorer remains the primary representation-based evaluator. This section keeps a single Moving MNIST example in the notebook so we can compare the real-video demo against a known synthetic control.

In [ ]:
import json

if load_error is not None:
    raise RuntimeError(f'Cannot score examples because V-JEPA failed to load: {load_error}')

vjepa_bundle = vjepa_scorer.score_example(
    example.observed,
    example.candidates,
    candidate_metadata=example.metadata['candidate_strategies'],
)
vjepa_candidate_table = build_candidate_score_table(vjepa_bundle, correct_index=example.correct_index)
vjepa_ranking_summary = build_ranking_summary(vjepa_bundle, correct_index=example.correct_index)
vjepa_confidence_tier = getattr(vjepa_bundle, 'confidence_tier', getattr(vjepa_bundle, 'uncertainty', 'unknown'))

single_example_scores = {
    'example_index': example.metadata['index'],
    'correct_index': int(example.correct_index),
    'runtime_info': runtime_info,
    'vjepa_score_bundle': vjepa_bundle.as_dict(),
    'candidate_score_table': vjepa_candidate_table,
    'ranking_summary': vjepa_ranking_summary,
}

single_example_scores_path = vjepa_results_dir / 'single_example_scores.json'
single_example_scores_path.write_text(json.dumps(single_example_scores, indent=2), encoding='utf-8')

print(f'Correct candidate index: {example.correct_index}')
print(f'V-JEPA selected index: {vjepa_bundle.selected_index}')
print(f'Confidence margin: {vjepa_bundle.confidence:.4f} ({vjepa_confidence_tier})')
print('\nCandidate score table:')
for row in vjepa_candidate_table:
    print(json.dumps(row, indent=2))
print(f'\nSaved V-JEPA score bundle to: {single_example_scores_path}')

In [ ]:
import json
from IPython.display import Markdown, display

baseline_result = run_future_selection_pipeline(
    observed=example.observed,
    candidates=example.candidates,
    candidate_metadata=example.metadata['candidate_strategies'],
    evaluation_mode='heuristic',
)

vjepa_pipeline_result = run_future_selection_pipeline(
    observed=example.observed,
    candidates=example.candidates,
    candidate_metadata=example.metadata['candidate_strategies'],
    representation_model=vjepa_scorer,
    evaluation_mode='representation_only',
)

comparison_table = build_baseline_comparison_table(
    vjepa_bundle,
    baseline_result,
    correct_index=example.correct_index,
)
context_payload = build_future_selection_context(
    example,
    vjepa_bundle,
    baseline_bundle=baseline_result,
)

vjepa_commentary = commentary_generator.generate(context_payload)
vjepa_llm_context = llm_ready_builder.build(context_payload)

single_example_trace = {
    'example_index': example.metadata['index'],
    'correct_index': int(example.correct_index),
    'vjepa_pipeline_result': vjepa_pipeline_result.as_dict(),
    'heuristic_pipeline_result': baseline_result.as_dict(),
    'comparison_table': comparison_table,
    'context_payload': context_payload,
    'commentary': vjepa_commentary.as_dict(),
    'llm_ready_commentary_context': vjepa_llm_context.as_dict(),
}

single_example_trace_path = vjepa_results_dir / 'single_example_trace.json'
single_example_trace_path.write_text(json.dumps(single_example_trace, indent=2), encoding='utf-8')
(vjepa_results_dir / 'single_example_commentary.json').write_text(
    json.dumps(vjepa_commentary.as_dict(), indent=2),
    encoding='utf-8',
)
(vjepa_results_dir / 'single_example_commentary.md').write_text(
    vjepa_commentary.to_markdown() + '\n',
    encoding='utf-8',
)
(vjepa_results_dir / 'llm_ready_commentary_context.json').write_text(
    json.dumps(vjepa_llm_context.as_dict(), indent=2),
    encoding='utf-8',
)

print('Heuristic vs engineered V-JEPA comparison on the control benchmark:')
for row in comparison_table:
    print(json.dumps(row, indent=2))
print(f'\nSaved comparison trace to: {single_example_trace_path}')
print('\nGrounded commentary:')
display(Markdown(vjepa_commentary.to_markdown()))

## 5. Real-World Dataset Preparation and SSV2-Aligned V-JEPA Setup

This section keeps scope tight: it targets Something-Something V2 only, rebuilds contiguous local 16-frame task clips, switches the real-video branch to the official SSV2-aligned V-JEPA checkpoint, and scores one example with both the old masked-only path and the new boundary-focused frozen hybrid.


In [ ]:
import json
from IPython.display import Image as NotebookImage, display
from PIL import Image, ImageDraw
import torch

if load_error is not None:
    raise RuntimeError(f'Cannot run the real-video smoke path because the control V-JEPA load failed: {load_error}')

real_vjepa_config = make_ssv2_vjepa_config(base=adapter_config, cache_dir=str(HF_CACHE_DIR))
real_vjepa_adapter = VJEPA2Adapter(real_vjepa_config)
real_vjepa_runtime = None
try:
    real_vjepa_runtime = real_vjepa_adapter.load()
except Exception as exc:
    raise RuntimeError(
        'Failed to load the SSV2-aligned V-JEPA checkpoint. The smoke gate should not continue on the generic control backbone.'
    ) from exc

real_masked_only_scorer = VJEPAFutureScorer(
    adapter=real_vjepa_adapter,
    scoring_variant='masked_future_prediction',
    auto_route_real_video=False,
)
real_hybrid_scorer = VJEPAFutureScorer(
    adapter=real_vjepa_adapter,
    scoring_variant='masked_boundary_hybrid',
    auto_route_real_video=False,
)

real_adapter = SomethingSomethingV2SubsetAdapter(real_video_config)
real_eval_split = os.environ.get('JEPA_REAL_VIDEO_EVAL_SPLIT', 'test')
force_rebuild_real_cache = os.environ.get('JEPA_REAL_VIDEO_FORCE_REBUILD', '0').strip().lower() in {'1', 'true', 'yes'}
real_video_load_error = None
real_eval_manifest = None
real_eval_dataset = None
real_example = None
real_example_panel_path = real_video_results_dir / 'single_example_panel.png'

def _frame_to_rgb(frame):
    frame = torch.as_tensor(frame).detach().cpu().clamp(0.0, 1.0)
    if frame.ndim != 3:
        raise ValueError(f'Expected frame shape (C, H, W); got {tuple(frame.shape)}')
    if frame.shape[0] == 1:
        frame = frame.repeat(3, 1, 1)
    if frame.shape[0] != 3:
        raise ValueError(f'Expected 1 or 3 channels; got {frame.shape[0]}')
    array = frame.mul(255).to(torch.uint8).permute(1, 2, 0).numpy()
    return Image.fromarray(array, mode='RGB')

def _sequence_to_grid(sequence, label, columns=4, padding=2):
    frames = [_frame_to_rgb(frame) for frame in sequence]
    width, height = frames[0].size
    columns = max(1, min(columns, len(frames)))
    rows = (len(frames) + columns - 1) // columns
    grid = Image.new('RGB', (columns * width + padding * (columns - 1), rows * height + padding * (rows - 1)), color=(0, 0, 0))
    for frame_index, frame in enumerate(frames):
        row = frame_index // columns
        column = frame_index % columns
        x = column * (width + padding)
        y = row * (height + padding)
        grid.paste(frame, (x, y))
    canvas = Image.new('RGB', (grid.width, grid.height + 22), color=(0, 0, 0))
    canvas.paste(grid, (0, 22))
    draw = ImageDraw.Draw(canvas)
    draw.text((4, 3), label, fill=(255, 255, 255))
    return canvas

def _save_real_example_panel(example, path):
    panels = [_sequence_to_grid(example.observed, 'observed (0:8)')]
    for candidate_index, candidate in enumerate(example.candidates):
        meta = example.metadata['candidate_strategies'][candidate_index]
        label = f"candidate {candidate_index} | {meta.get('strategy', 'unknown')}"
        panels.append(_sequence_to_grid(candidate, label))
    width = max(panel.width for panel in panels)
    height = sum(panel.height for panel in panels) + 4 * (len(panels) - 1)
    canvas = Image.new('RGB', (width, height), color=(0, 0, 0))
    y = 0
    for panel in panels:
        canvas.paste(panel, (0, y))
        y += panel.height + 4
    canvas.save(path)
    return path

try:
    real_eval_manifest = real_adapter.prepare_cache(real_eval_split, force_rebuild=force_rebuild_real_cache)
    real_eval_dataset = real_adapter.build_dataset(real_eval_split)
    real_example = real_eval_dataset[0]
    _save_real_example_panel(real_example, real_example_panel_path)
except Exception as exc:
    real_video_load_error = exc

if real_video_load_error is not None:
    raise RuntimeError(
        'Something-Something V2 ingest failed for the SSV2 smoke path. Stop here and inspect the exact blocker instead of switching datasets.'
    ) from real_video_load_error

print('SSV2-aligned real-video setup:')
print(json.dumps(real_vjepa_config.as_dict(), indent=2))
print('Real-video cache manifest:')
print(f'- dataset source: {real_adapter.config.dataset_name}')
print(f'- eval split: {real_eval_split}')
print(f'- eval manifest: {real_eval_manifest}')
print(f'- eval examples: {len(real_eval_dataset)}')
display(NotebookImage(filename=str(real_example_panel_path)))

real_masked_bundle = real_masked_only_scorer.score_example(
    real_example.observed,
    real_example.candidates,
    candidate_metadata=real_example.metadata['candidate_strategies'],
)
real_hybrid_bundle = real_hybrid_scorer.score_example(
    real_example.observed,
    real_example.candidates,
    candidate_metadata=real_example.metadata['candidate_strategies'],
)
real_baseline_result = run_future_selection_pipeline(
    observed=real_example.observed,
    candidates=real_example.candidates,
    candidate_metadata=real_example.metadata['candidate_strategies'],
    evaluation_mode='heuristic',
)
real_single_example_comparison = build_multi_evaluator_comparison_table(
    {
        'heuristic': real_baseline_result,
        'masked_only': real_masked_bundle,
        'boundary_hybrid': real_hybrid_bundle,
    },
    correct_index=real_example.correct_index,
)

(real_video_results_dir / 'single_example_scores.json').write_text(
    json.dumps(
        {
            'source_video_id': real_example.metadata['source_video_id'],
            'correct_index': int(real_example.correct_index),
            'control_runtime_info': runtime_info,
            'real_vjepa_runtime_info': real_vjepa_runtime,
            'dataset_source': real_adapter.config.dataset_name,
            'real_vjepa_config': real_vjepa_config.as_dict(),
            'masked_only_bundle': real_masked_bundle.as_dict(),
            'boundary_hybrid_bundle': real_hybrid_bundle.as_dict(),
        },
        indent=2,
    ),
    encoding='utf-8',
)
(real_video_results_dir / 'single_example_trace.json').write_text(
    json.dumps({'comparison_table': real_single_example_comparison}, indent=2),
    encoding='utf-8',
)

print('Single-example real-video comparison:')
for row in real_single_example_comparison:
    print(json.dumps(row, indent=2))


## 6. Tiny Frozen-Feature Reranker

This optional section fits a tiny reranker on top of the frozen real-video evaluators. It stays additive, reuses the current scorer outputs, and now defaults to a deterministic provisional split on the current SSV2 notebook path so the reranker can run end to end without extra setup.

In [ ]:
import os
import json
import numpy as np
from torch.utils.data import Subset

from jepa.scoring import FrozenFeatureRerankerConfig, FrozenFeatureRerankerScorer

reranker_results_dir = REPO_ROOT / 'results' / 'real_video_reranker_eval'
reranker_results_dir.mkdir(parents=True, exist_ok=True)

reranker_enabled = os.environ.get('JEPA_REAL_VIDEO_RERANKER_ENABLED', '1').strip().lower() in {'1', 'true', 'yes'}
reranker_allow_provisional = os.environ.get('JEPA_REAL_VIDEO_RERANKER_ALLOW_PROVISIONAL', '1').strip().lower() in {'1', 'true', 'yes'}
reranker_include_candidate_type_indicators = os.environ.get(
    'JEPA_REAL_VIDEO_RERANKER_INCLUDE_CANDIDATE_TYPE_INDICATORS',
    '0',
).strip().lower() in {'1', 'true', 'yes'}
reranker_model_kind = os.environ.get('JEPA_REAL_VIDEO_RERANKER_MODEL_KIND', 'linear').strip().lower()
reranker_train_split = os.environ.get('JEPA_REAL_VIDEO_RERANKER_TRAIN_SPLIT', 'train')
reranker_eval_split = os.environ.get('JEPA_REAL_VIDEO_RERANKER_EVAL_SPLIT', real_eval_split)
reranker_train_fraction = float(os.environ.get('JEPA_REAL_VIDEO_RERANKER_TRAIN_FRACTION', '0.75'))
reranker_eval_count_env = int(os.environ.get('JEPA_REAL_VIDEO_RERANKER_EVAL_COUNT', str(len(real_eval_dataset))))
reranker_train_max_examples = int(os.environ.get('JEPA_REAL_VIDEO_RERANKER_MAX_TRAIN', '128'))
reranker_seed = int(os.environ.get('JEPA_REAL_VIDEO_RERANKER_SEED', '17'))
reranker_force_rebuild_cache = os.environ.get('JEPA_REAL_VIDEO_RERANKER_FORCE_REBUILD', '0').strip().lower() in {'1', 'true', 'yes'}
reranker_evaluator_name = f'tiny_reranker_{reranker_model_kind}'

reranker_training_summary = None
reranker_benchmark_result = None
reranker_feature_spec = None
reranker_blocker_summary = {
    'enabled': bool(reranker_enabled),
    'allow_provisional_overlap_split': bool(reranker_allow_provisional),
    'include_heuristic_features': True,
    'include_candidate_type_indicators': bool(reranker_include_candidate_type_indicators),
    'model_kind': reranker_model_kind,
    'train_split': reranker_train_split,
    'eval_split': reranker_eval_split,
}

if not reranker_enabled:
    reranker_blocker_summary['status'] = 'disabled'
    reranker_blocker_summary['reason'] = 'JEPA_REAL_VIDEO_RERANKER_ENABLED is disabled.'
    (reranker_results_dir / 'reranker_blocker_summary.json').write_text(
        json.dumps(reranker_blocker_summary, indent=2),
        encoding='utf-8',
    )
    print('\nReranker section skipped because JEPA_REAL_VIDEO_RERANKER_ENABLED is disabled.')
else:
    reranker_train_source_split = real_adapter.source_split_for_request(reranker_train_split)
    reranker_eval_source_split = real_adapter.source_split_for_request(reranker_eval_split)
    reranker_blocker_summary['train_source_split'] = reranker_train_source_split
    reranker_blocker_summary['eval_source_split'] = reranker_eval_source_split
    reranker_blocker_summary['strict_split_requested'] = True

    strict_split_blocked = reranker_train_source_split == reranker_eval_source_split
    reranker_train_dataset = None
    reranker_eval_dataset_obj = None
    reranker_mode = 'strict'
    provisional_mode = False
    train_cache_split = 'train'
    eval_cache_split = 'eval'

    if strict_split_blocked and not reranker_allow_provisional:
        reranker_blocker_summary['status'] = 'blocked'
        reranker_blocker_summary['reason'] = (
            f"Strict reranker split unavailable: train split {reranker_train_split!r} and eval split {reranker_eval_split!r} "
            f"both resolve to source split {reranker_eval_source_split!r}. Set JEPA_REAL_VIDEO_RERANKER_ALLOW_PROVISIONAL=1 "
            'to use a deterministic provisional split inside the cached pool.'
        )
        (reranker_results_dir / 'reranker_blocker_summary.json').write_text(
            json.dumps(reranker_blocker_summary, indent=2),
            encoding='utf-8',
        )
        print('\nReranker blocked by strict split safety:')
        print(json.dumps(reranker_blocker_summary, indent=2))
    else:
        if strict_split_blocked:
            if len(real_eval_dataset) < 2:
                reranker_blocker_summary['status'] = 'blocked'
                reranker_blocker_summary['reason'] = 'Need at least two cached real-video examples for a provisional reranker split.'
                (reranker_results_dir / 'reranker_blocker_summary.json').write_text(
                    json.dumps(reranker_blocker_summary, indent=2),
                    encoding='utf-8',
                )
                print('\nReranker blocked because the cached pool is too small for a provisional split:')
                print(json.dumps(reranker_blocker_summary, indent=2))
            else:
                pool_size = len(real_eval_dataset)
                rng = np.random.default_rng(reranker_seed)
                permutation = rng.permutation(pool_size)
                train_count = max(1, min(pool_size - 1, int(round(pool_size * reranker_train_fraction))))
                train_indices = permutation[:train_count].tolist()
                eval_indices = permutation[train_count:].tolist()
                reranker_train_dataset = Subset(real_eval_dataset, train_indices)
                reranker_eval_dataset_obj = Subset(real_eval_dataset, eval_indices)
                reranker_mode = 'provisional'
                provisional_mode = True
                train_cache_split = 'provisional_train'
                eval_cache_split = 'provisional_eval'
                reranker_blocker_summary['status'] = 'provisional'
                reranker_blocker_summary['reason'] = 'Using a deterministic provisional split inside the cached real-video pool.'
                reranker_blocker_summary['train_examples'] = len(reranker_train_dataset)
                reranker_blocker_summary['eval_examples'] = len(reranker_eval_dataset_obj)
        else:
            reranker_train_dataset = real_adapter.build_dataset(reranker_train_split)
            reranker_eval_dataset_obj = real_adapter.build_dataset(reranker_eval_split)
            reranker_blocker_summary['status'] = 'strict'
            reranker_blocker_summary['reason'] = 'Using a clean train/eval split resolved by the adapter.'
            reranker_blocker_summary['train_examples'] = len(reranker_train_dataset)
            reranker_blocker_summary['eval_examples'] = len(reranker_eval_dataset_obj)

        if reranker_train_dataset is not None and reranker_eval_dataset_obj is not None:
            reranker_config = FrozenFeatureRerankerConfig(
                model_kind=reranker_model_kind,
                include_heuristic_features=True,
                include_candidate_type_indicators=reranker_include_candidate_type_indicators,
                strict_split=not provisional_mode,
                allow_provisional_overlap_split=reranker_allow_provisional,
                seed=reranker_seed,
                cache_dir=str(reranker_results_dir),
            ).validate()
            reranker_scorer = FrozenFeatureRerankerScorer(
                reranker_config,
                masked_only_scorer=real_masked_only_scorer,
                boundary_hybrid_scorer=real_hybrid_scorer,
            )

            reranker_train_max_examples = min(reranker_train_max_examples, len(reranker_train_dataset))
            reranker_eval_count = min(reranker_eval_count_env, len(reranker_eval_dataset_obj))

            train_cache = reranker_scorer.build_feature_cache(
                reranker_train_dataset,
                split_name=train_cache_split,
                max_examples=reranker_train_max_examples,
                provisional=provisional_mode,
                force_rebuild=reranker_force_rebuild_cache,
            )
            eval_cache = reranker_scorer.build_feature_cache(
                reranker_eval_dataset_obj,
                split_name=eval_cache_split,
                max_examples=len(reranker_eval_dataset_obj),
                provisional=provisional_mode,
                force_rebuild=reranker_force_rebuild_cache,
            )
            reranker_feature_spec = {
                **reranker_scorer.feature_spec(),
                'train_cache': train_cache.as_dict(),
                'eval_cache': eval_cache.as_dict(),
                'reranker_mode': reranker_mode,
                'train_source_split': reranker_train_source_split,
                'eval_source_split': reranker_eval_source_split,
            }
            (reranker_results_dir / 'reranker_feature_spec.json').write_text(
                json.dumps(reranker_feature_spec, indent=2),
                encoding='utf-8',
            )

            reranker_training_summary = reranker_scorer.fit(
                reranker_train_dataset,
                max_examples=reranker_train_max_examples,
                split_name=train_cache_split,
                provisional=provisional_mode,
                force_rebuild_cache=reranker_force_rebuild_cache,
            ).as_dict()
            reranker_training_summary.update(
                {
                    'reranker_mode': reranker_mode,
                    'train_source_split': reranker_train_source_split,
                    'eval_source_split': reranker_eval_source_split,
                    'provisional_mode': provisional_mode,
                    'candidate_type_indicators': reranker_include_candidate_type_indicators,
                }
            )
            (reranker_results_dir / 'reranker_training_summary.json').write_text(
                json.dumps(reranker_training_summary, indent=2),
                encoding='utf-8',
            )
            (reranker_results_dir / 'reranker_blocker_summary.json').write_text(
                json.dumps(reranker_blocker_summary, indent=2),
                encoding='utf-8',
            )

            print('\nTiny frozen-feature reranker summary:')
            print(json.dumps(reranker_training_summary, indent=2))
            print('\nReranker evaluation defaults:')
            print(
                f"evaluation_count={reranker_eval_count}, mode={reranker_mode}, "
                f"include_candidate_type_indicators={reranker_include_candidate_type_indicators}."
            )

            reranker_benchmark_result = run_future_selection_benchmark(
                reranker_eval_dataset_obj,
                evaluators={
                    'heuristic': None,
                    'masked_only': real_masked_only_scorer,
                    'boundary_hybrid': real_hybrid_scorer,
                    reranker_evaluator_name: reranker_scorer,
                },
                evaluation_count=reranker_eval_count,
                show_progress=True,
                progress_interval=1,
            )
            reranker_artifacts = save_future_selection_benchmark_artifacts(
                reranker_benchmark_result,
                reranker_results_dir,
                report_path=reranker_results_dir / 'reranker_eval_summary.md',
                title='Tiny Frozen-Feature Reranker Evaluation Summary',
                metadata={
                    'dataset_name': real_video_config.dataset_name,
                    'train_split': reranker_train_split,
                    'eval_split': reranker_eval_split,
                    'train_source_split': reranker_train_source_split,
                    'eval_source_split': reranker_eval_source_split,
                    'reranker_mode': reranker_mode,
                    'include_candidate_type_indicators': reranker_include_candidate_type_indicators,
                    'include_heuristic_features': True,
                    'model_kind': reranker_model_kind,
                    'feature_dim': reranker_feature_spec['train_cache']['features_shape'][-1],
                    'confusion_report': reranker_benchmark_result.confusion_report,
                },
            )
            print('\nTiny frozen-feature reranker benchmark summary:')
            print(json.dumps(reranker_benchmark_result.summary, indent=2))
            print('\nTiny frozen-feature reranker confusion report:')
            print(json.dumps(reranker_benchmark_result.confusion_report, indent=2))
            print('\nSaved reranker artifacts:')
            for name, artifact_path in reranker_artifacts.items():
                print(f'- {name}: {artifact_path}')


## 7. SSV2 Smoke Gate

This section runs only the 8-example smoke gate. It compares heuristic, masked-only V-JEPA, and the new boundary-focused frozen hybrid, then stops immediately if the hybrid path does not clearly improve over the old masked-only real-video scorer.

In [ ]:
import json

if real_video_load_error is not None or real_eval_dataset is None:
    raise RuntimeError('Real-video smoke evaluation is unavailable because dataset preparation failed earlier.')

real_smoke_count = min(int(os.environ.get('JEPA_REAL_VIDEO_SMOKE_COUNT', '8')), len(real_eval_dataset))
print(
    f'SSV2 smoke-gate defaults: evaluation_count={real_smoke_count}. '
    'Set JEPA_REAL_VIDEO_SMOKE_COUNT for a different slice.'
)

real_benchmark_result = run_future_selection_benchmark(
    real_eval_dataset,
    evaluators={
        'heuristic': None,
        'masked_only': real_masked_only_scorer,
        'boundary_hybrid': real_hybrid_scorer,
    },
    evaluation_count=real_smoke_count,
    show_progress=True,
    progress_interval=1,
)
real_benchmark_artifacts = save_future_selection_benchmark_artifacts(
    real_benchmark_result,
    real_video_results_dir,
    report_path=report_dir / 'real_video_eval_summary.md',
    title='SSV2 Smoke-Gate Evaluation Summary',
    metadata={
        'dataset_name': real_video_config.dataset_name,
        'eval_split': real_eval_split,
        'source_window_length': real_video_config.source_window_length,
        'real_vjepa_model_id': real_vjepa_config.model_id,
        'real_vjepa_target_frames': real_vjepa_config.target_frames,
        'confusion_report': real_benchmark_result.confusion_report,
    },
)

summary = real_benchmark_result.summary
per_negative = real_benchmark_result.per_negative_type
confusions = real_benchmark_result.confusion_report
masked_summary = summary['masked_only']
hybrid_summary = summary['boundary_hybrid']

def _confusion_count(evaluator_name, strategy_names):
    payload = confusions.get(evaluator_name, {})
    return sum(int(payload.get(name, {}).get('count', 0)) for name in strategy_names)

def _weighted_negative_accuracy(evaluator_name, strategy_names):
    payload = per_negative.get(evaluator_name, {})
    matched = []
    for name in strategy_names:
        strategy_payload = payload.get(name)
        if strategy_payload is None:
            continue
        matched.append(
            (
                int(strategy_payload.get('count', 0)),
                float(strategy_payload.get('accuracy_when_present', 0.0)),
            )
        )
    total = sum(count for count, _ in matched)
    if total <= 0:
        return None
    weighted_correct = sum(count * accuracy for count, accuracy in matched)
    return float(weighted_correct / total)

temporal_negative_names = {
    'temporal_order_two_block_swap',
    'temporal_order_block_reverse',
    'temporal_order_reverse',
    'shuffled_temporal_order',
}
same_template_names = {'future_segment_from_other_sample'}

masked_temporal_accuracy = _weighted_negative_accuracy('masked_only', temporal_negative_names)
hybrid_temporal_accuracy = _weighted_negative_accuracy('boundary_hybrid', temporal_negative_names)
masked_same_template_accuracy = _weighted_negative_accuracy('masked_only', same_template_names)
hybrid_same_template_accuracy = _weighted_negative_accuracy('boundary_hybrid', same_template_names)
masked_temporal_confusions = _confusion_count('masked_only', temporal_negative_names)
hybrid_temporal_confusions = _confusion_count('boundary_hybrid', temporal_negative_names)
masked_same_template_confusions = _confusion_count('masked_only', same_template_names)
hybrid_same_template_confusions = _confusion_count('boundary_hybrid', same_template_names)

smoke_gate_reasons = []
if hybrid_summary['top1_accuracy'] <= masked_summary['top1_accuracy']:
    smoke_gate_reasons.append(
        f"boundary_hybrid accuracy {hybrid_summary['top1_accuracy']:.4f} did not beat masked_only {masked_summary['top1_accuracy']:.4f}"
    )
if hybrid_summary['score_margin_mean'] <= masked_summary['score_margin_mean']:
    smoke_gate_reasons.append(
        f"boundary_hybrid score margin mean {hybrid_summary['score_margin_mean']:.4f} did not beat masked_only {masked_summary['score_margin_mean']:.4f}"
    )
if hybrid_temporal_accuracy is None or masked_temporal_accuracy is None:
    smoke_gate_reasons.append('Temporal-negative accuracy was unavailable for the smoke gate comparison.')
elif hybrid_temporal_accuracy <= masked_temporal_accuracy:
    smoke_gate_reasons.append(
        f"boundary_hybrid temporal-negative accuracy {hybrid_temporal_accuracy:.4f} did not beat masked_only {masked_temporal_accuracy:.4f}"
    )
if hybrid_same_template_accuracy is None or masked_same_template_accuracy is None:
    smoke_gate_reasons.append('Same-template accuracy was unavailable for the smoke gate comparison.')
elif hybrid_same_template_accuracy <= masked_same_template_accuracy:
    smoke_gate_reasons.append(
        f"boundary_hybrid same-template accuracy {hybrid_same_template_accuracy:.4f} did not beat masked_only {masked_same_template_accuracy:.4f}"
    )
if hybrid_temporal_confusions > masked_temporal_confusions:
    smoke_gate_reasons.append(
        f'boundary_hybrid temporal-order confusions {hybrid_temporal_confusions} exceeded masked_only {masked_temporal_confusions}'
    )
if hybrid_same_template_confusions > masked_same_template_confusions:
    smoke_gate_reasons.append(
        f'boundary_hybrid same-template confusions {hybrid_same_template_confusions} exceeded masked_only {masked_same_template_confusions}'
    )

smoke_gate_passed = len(smoke_gate_reasons) == 0
smoke_gate_summary = {
    'smoke_count': real_smoke_count,
    'passed': smoke_gate_passed,
    'reasons': smoke_gate_reasons,
    'masked_only_summary': masked_summary,
    'boundary_hybrid_summary': hybrid_summary,
    'heuristic_summary': summary['heuristic'],
    'masked_temporal_accuracy': masked_temporal_accuracy,
    'boundary_temporal_accuracy': hybrid_temporal_accuracy,
    'masked_same_template_accuracy': masked_same_template_accuracy,
    'boundary_same_template_accuracy': hybrid_same_template_accuracy,
    'masked_temporal_confusions': masked_temporal_confusions,
    'boundary_temporal_confusions': hybrid_temporal_confusions,
    'masked_same_template_confusions': masked_same_template_confusions,
    'boundary_same_template_confusions': hybrid_same_template_confusions,
    'confusion_report': confusions,
    'per_negative_type': per_negative,
}
(real_video_results_dir / 'smoke_gate_summary.json').write_text(
    json.dumps(smoke_gate_summary, indent=2),
    encoding='utf-8',
)

print('SSV2 smoke benchmark summary:')
print(json.dumps(summary, indent=2))
print('\nPer-negative-type summary:')
print(json.dumps(per_negative, indent=2))
print('\nFocused confusion report:')
print(json.dumps(confusions, indent=2))
print('\nSaved smoke-gate artifacts:')
for name, path in real_benchmark_artifacts.items():
    print(f'- {name}: {path}')

reranker_status_path = reranker_results_dir / 'reranker_blocker_summary.json'
if reranker_status_path.exists():
    print('\nReranker status summary:')
    print(reranker_status_path)
    print(reranker_status_path.read_text(encoding='utf-8'))

if not smoke_gate_passed:
    print('\nSSV2 smoke gate failed for these reasons:')
    for reason in smoke_gate_reasons:
        print(f'- {reason}')
    print('\nSmoke gate failure means the current frozen scorer path did not clear the quality bar; it is not a Colab/runtime crash. The notebook continues so you can inspect reranker outputs and saved artifacts.')
else:
    print('\nSSV2 smoke gate passed.')


## 8. Saved Artifacts and Export

This final section surfaces the Moving MNIST control artifacts, the reranker artifacts when present, and the real-video artifacts, then packages everything into one archive so results can be pulled back from the remote runtime reliably.

In [ ]:
import os
os.environ['JEPA_DRIVE_EXPORT_DIR'] = '/content/drive/MyDrive/jepa_exports'
# Optional for PyCharm-connected Colab kernels: disable browser download and rely on Drive copy.
os.environ.setdefault('JEPA_DOWNLOAD_ARTIFACT_BUNDLE', '0')

In [ ]:
from pathlib import Path
from datetime import datetime
from IPython.display import Image as NotebookImage, display
import json
import os
import shutil
import zipfile

artifact_export_dir = REPO_ROOT / 'results' / 'artifact_exports'
artifact_export_dir.mkdir(parents=True, exist_ok=True)

artifact_roots = [
    REPO_ROOT / 'results' / 'moving_mnist',
    REPO_ROOT / 'results' / 'task_examples',
    REPO_ROOT / 'results' / 'vjepa_eval',
    REPO_ROOT / 'results' / 'real_video_eval',
    REPO_ROOT / 'results' / 'real_video_reranker_eval',
    REPO_ROOT / 'results' / 'agent_live',
    REPO_ROOT / 'report',
]

for root in artifact_roots:
    print(f'Artifacts under {root}:')
    if not root.exists():
        print('  - missing')
        continue
    for path in sorted(root.iterdir()):
        print(f'  - {path.name}')

panel_paths = [
    *(task_results_dir.glob('*_panel.png')),
    *(real_video_results_dir.glob('*panel*.png')),
]
if panel_paths:
    display(NotebookImage(filename=str(sorted(panel_paths)[0])))

commentary_paths = [
    vjepa_results_dir / 'single_example_commentary.md',
    real_video_results_dir / 'single_example_commentary.md',
    reranker_results_dir / 'reranker_eval_summary.md',
    report_dir / 'vjepa_eval_summary.md',
    report_dir / 'real_video_eval_summary.md',
]
latest_live_markdown = sorted(agent_live_results_dir.glob('*.md'))
if latest_live_markdown:
    commentary_paths.append(latest_live_markdown[-1])

for path in commentary_paths:
    if path.exists():
        print(f'\nRendered text from {path.name}:\n')
        print(path.read_text(encoding='utf-8'))

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
bundle_path = artifact_export_dir / f'jepa_demo_artifacts_{timestamp}.zip'
included_files = []
total_bytes = 0

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for root in artifact_roots:
        if not root.exists():
            continue
        for file_path in sorted(p for p in root.rglob('*') if p.is_file()):
            arcname = file_path.relative_to(REPO_ROOT).as_posix()
            bundle.write(file_path, arcname)
            included_files.append(arcname)
            total_bytes += int(file_path.stat().st_size)

bundle_summary = {
    'archive_path': str(bundle_path),
    'file_count': len(included_files),
    'total_bytes': total_bytes,
    'included_files': included_files,
}
print(f'\nCreated artifact bundle: {bundle_path}')
print(json.dumps(bundle_summary, indent=2))

drive_export_dir = os.environ.get('JEPA_DRIVE_EXPORT_DIR', '').strip()
if drive_export_dir:
    drive_target = Path(drive_export_dir).expanduser()
    if str(drive_target).startswith('/content/drive'):
        try:
            from google.colab import drive
            if not Path('/content/drive/MyDrive').exists():
                print('Mounting Google Drive for artifact export...')
                drive.mount('/content/drive', force_remount=False)
        except Exception as exc:
            print(f'Unable to mount Google Drive automatically: {exc}')
    drive_target.mkdir(parents=True, exist_ok=True)
    copied_path = drive_target / bundle_path.name
    shutil.copy2(bundle_path, copied_path)
    print(f'Copied artifact bundle to: {copied_path}')

try:
    from google.colab import files
    if os.environ.get('JEPA_DOWNLOAD_ARTIFACT_BUNDLE', '1').strip().lower() in {'1', 'true', 'yes'}:
        print('Attempting browser download of the artifact bundle...')
        files.download(str(bundle_path))
except Exception as exc:
    print(f'Automatic download unavailable in this notebook client: {exc}')

if not drive_export_dir:
    print('No JEPA_DRIVE_EXPORT_DIR is set. In PyCharm-connected Colab sessions, browser download may not bring the file back to your local machine.')
    print('Set JEPA_DRIVE_EXPORT_DIR to a mounted Drive folder if you want a reliable export path.')
